# RQ2/RQ3/RQ4 - Selection, Efficiency, and Ablation

This notebook rebuilds the tables and runtime figure from released CSV files.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS = ROOT / 'results'
FIGURES = ROOT / 'figures'

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 300,
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

def save_fig(fig, name):
    FIGURES.mkdir(exist_ok=True)
    fig.savefig(FIGURES / f'{name}.png', bbox_inches='tight')
    fig.savefig(FIGURES / f'{name}.pdf', bbox_inches='tight')

## Selected Reduced Meta-features

In [ ]:
selected = pd.read_csv(RESULTS / 'rq2_meta_feature_selection/selected_features_pearson085_random_forest_importance_k10.csv')
display(selected)
family_counts = selected['family'].value_counts().reindex(['syn', 'cls', 'tda'], fill_value=0)
fig, ax = plt.subplots(figsize=(5.5, 3))
colors = ['#2CA02C', '#1F77B4', '#FF7F0E']
ax.bar(family_counts.index, family_counts.values, color=colors)
ax.set_ylabel('# meta-features')
ax.grid(axis='y', alpha=0.25)
for i, v in enumerate(family_counts.values):
    ax.text(i, v + 0.1, str(int(v)), ha='center')
fig.tight_layout()
plt.show()

## Full vs Reduced Performance

In [ ]:
perf = pd.read_csv(RESULTS / 'rq2_meta_feature_selection/rq2_rq3_table_for_paper.csv')
cols = ['config', 'feature_set', 'n_features', 'mean_f1', 'std_f1', 'mean_precision', 'std_precision', 'mean_recall', 'std_recall']
display(perf[cols].round(2))

## Runtime Figure

In [ ]:
eff = pd.read_csv(RESULTS / 'rq3_efficiency/table_efficiency_preliminary_application_total_551_pairs_with_reduced.csv')
plot_df = eff.sort_values('known_total_time_sec_551_pairs', ascending=True).copy()
colors = ['#1F77B4' if 'MetaMatch' in m else '#A6A6A6' for m in plot_df['method']]
fig, ax = plt.subplots(figsize=(8, 5.5))
bars = ax.barh(plot_df['method'], plot_df['known_total_time_hours_551_pairs'], color=colors)
ax.set_xlabel('Total runtime (hours)')
ax.grid(axis='x', alpha=0.25)
for bar, value in zip(bars, plot_df['known_total_time_hours_551_pairs']):
    ax.text(value + max(plot_df['known_total_time_hours_551_pairs']) * 0.01, bar.get_y() + bar.get_height()/2, f'{value:.2f}', va='center', fontsize=8)
fig.tight_layout()
save_fig(fig, 'fig_rq4_total_runtime_bar')
plt.show()
display(eff.round(2))

## Family Ablation

In [ ]:
ablation = pd.read_csv(RESULTS / 'rq4_family_ablation/family_ablation_complete_vs_reduced_for_paper.csv')
display(ablation.round(2))

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(ablation))
w = 0.35
ax.bar(x - w/2, ablation['complete_f1_mean'], width=w, yerr=ablation['complete_f1_std'], label='Complete', color='#1F77B4', alpha=0.85)
ax.bar(x + w/2, ablation['reduced_f1_mean'], width=w, yerr=ablation['reduced_f1_std'], label='Reduced', color='#FF7F0E', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(ablation['feature_set'], rotation=30, ha='right')
ax.set_ylabel('F1')
ax.set_ylim(0, 1.05)
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
plt.show()